# 05. SCM Decision Brief

## Objective

`mart_scm_decision_weekly`의 주간 의사결정 결과를
SCM 담당자가 바로 확인할 수 있는 운영형 Brief로 변환한다.

본 Brief는 다음 질문에 답한다.

1. 현재 시장 가격 Risk는 어떤 상태인가?
2. 다음 주 가격 Risk는 어떻게 예상되는가?
3. 현재 Inventory Position은 충분한가?
4. Baseline 대비 발주량을 조정해야 하는가?
5. 조정한다면 왜 조정해야 하는가?

> 주의:
> 시장 가격 데이터는 실제 국내 공공데이터를 사용하지만,
> 수요·재고·입고·리드타임은 기업 내부 데이터가 없어
> Simulation으로 구성하였다.

In [1]:
from pathlib import Path
from importlib.util import find_spec
import os

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

load_dotenv(
    PROJECT_ROOT / ".env"
)

print("Project root:", PROJECT_ROOT)

Project root: /Users/cindy/meanvelop/retailops-ax


In [3]:
def get_env(*names, default=None):

    for name in names:
        value = os.getenv(name)

        if value not in (None, ""):
            return value

    return default


DB_HOST = get_env(
    "DB_HOST",
    "POSTGRES_HOST",
    default="localhost",
)

DB_PORT = get_env(
    "DB_PORT",
    "POSTGRES_PORT",
    default="5432",
)

DB_NAME = get_env(
    "DB_NAME",
    "POSTGRES_DB",
    default="retailops_ax",
)

DB_USER = get_env(
    "DB_USER",
    "POSTGRES_USER",
)

DB_PASSWORD = get_env(
    "DB_PASSWORD",
    "POSTGRES_PASSWORD",
)

In [4]:
if find_spec("psycopg") is not None:
    driver = "postgresql+psycopg"

elif find_spec("psycopg2") is not None:
    driver = "postgresql+psycopg2"

else:
    raise ImportError(
        "psycopg 또는 psycopg2가 필요합니다."
    )

In [5]:
database_url = URL.create(
    drivername=driver,
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME,
)

engine = create_engine(
    database_url,
    pool_pre_ping=True,
)

In [6]:
query = """
select *
from dbt.mart_scm_decision_weekly
order by decision_week
"""

decision = pd.read_sql(
    query,
    engine,
    parse_dates=["decision_week"],
)

print("Rows:", len(decision))
print(
    decision["decision_week"].min(),
    "~",
    decision["decision_week"].max()
)

decision.tail()

Rows: 54
2025-10-05 00:00:00 ~ 2026-10-11 00:00:00


,decision_week,product_id,product_name,current_price_index,forecast_price_index,current_risk_level,forecast_risk_level,beginning_inventory_kg,inbound_qty_kg,open_order_qty_kg,...,ending_inventory_kg,ending_days_of_cover,decision_signal,decision_reason,has_price_forecast,data_quality_flag,policy_version,market_data_scope,enterprise_data_scope,decision_scope
49,2026-09-13,P001,양파,1.182188,1.152421,Risk,Risk,7500.000000,7500.0,8000.0,...,2525.724959,3.573518,BASELINE_REPLENISHMENT,기본 Safety Stock 및 Lead Time 기반 정기 보충발주,True,OK,price_early_watch_v1,PUBLIC_ACTUAL,SIMULATED,SIMULATION
50,2026-09-20,P001,양파,1.231528,1.157785,Risk,Risk,10525.724959,8000.0,0.0,...,5478.729564,7.774557,BASELINE_REPLENISHMENT,기본 Safety Stock 및 Lead Time 기반 정기 보충발주,True,OK,price_early_watch_v1,PUBLIC_ACTUAL,SIMULATED,SIMULATION
51,2026-09-27,P001,양파,1.040231,1.101819,Normal,Watch,9978.729564,4500.0,0.0,...,4801.850494,6.820810,ADVANCE_PURCHASE,Normal→Watch 가격 Risk 선행신호; 2일분 재고 Buffer 반영,True,OK,price_early_watch_v1,PUBLIC_ACTUAL,SIMULATED,SIMULATION
52,2026-10-04,P001,양파,1.175733,1.095188,Risk,Watch,11301.850494,6500.0,0.0,...,6943.991131,9.853827,OFFSET_REDUCTION,이전 선행발주로 Inventory Position이 확보되어 후속 발주량 축소,True,OK,price_early_watch_v1,PUBLIC_ACTUAL,SIMULATED,SIMULATION
53,2026-10-11,P001,양파,1.138085,1.127643,Watch,Watch,10443.991131,3500.0,0.0,...,5622.990592,7.955678,BASELINE_REPLENISHMENT,기본 Safety Stock 및 Lead Time 기반 정기 보충발주,True,OK,price_early_watch_v1,PUBLIC_ACTUAL,SIMULATED,SIMULATION


In [7]:
latest = (
    decision
    .sort_values("decision_week")
    .iloc[-1]
)

latest

decision_week                                   2026-10-11 00:00:00
product_id                                                     P001
product_name                                                     양파
current_price_index                                        1.138085
forecast_price_index                                       1.127643
current_risk_level                                            Watch
forecast_risk_level                                           Watch
beginning_inventory_kg                                 10443.991131
inbound_qty_kg                                               3500.0
open_order_qty_kg                                               0.0
inventory_position_kg                                  10443.991131
forecast_weekly_demand_kg                               4947.527099
lead_time_days                                                   10
safety_stock_days                                                 7
risk_buffer_days                                

In [8]:
def get_risk_message(row):

    current = row["current_risk_level"]
    forecast = row["forecast_risk_level"]

    if not row["has_price_forecast"]:
        return (
            "가격 Risk Forecast가 없어 "
            "Baseline 재고정책을 적용합니다."
        )

    if (
        current == "Normal"
        and forecast == "Watch"
    ):
        return (
            "현재 가격은 정상 범위이나 "
            "다음 주 Watch 수준 상승 가능성이 감지되었습니다."
        )

    if (
        current == "Normal"
        and forecast == "Risk"
    ):
        return (
            "현재 가격은 정상 범위이나 "
            "다음 주 Risk 수준 진입 가능성이 감지되었습니다."
        )

    if (
        current == "Watch"
        and forecast == "Risk"
    ):
        return (
            "가격 Risk가 Watch에서 Risk 수준으로 "
            "확대될 가능성이 있습니다."
        )

    if current == "Risk":
        return (
            "현재 가격이 이미 Risk 수준에 있습니다. "
            "추가 선행구매보다 기존 확보재고 점검이 우선입니다."
        )

    return (
        "현재 뚜렷한 가격 Risk 확대 신호는 없습니다."
    )

In [9]:
def get_action_message(row):

    signal = row["decision_signal"]

    baseline_qty = (
        row["baseline_order_qty_kg"]
    )

    recommended_qty = (
        row["recommended_order_qty_kg"]
    )

    shift_qty = (
        row["order_timing_shift_kg"]
    )


    if signal == "ADVANCE_PURCHASE":

        return (
            f"Baseline {baseline_qty:,.0f}kg 대비 "
            f"{shift_qty:,.0f}kg을 선행하여 "
            f"총 {recommended_qty:,.0f}kg 발주를 권고합니다."
        )


    if signal == "OFFSET_REDUCTION":

        return (
            f"이전 선행발주 물량을 반영하여 "
            f"Baseline 대비 {abs(shift_qty):,.0f}kg 감축한 "
            f"{recommended_qty:,.0f}kg 발주를 권고합니다."
        )


    if signal == "EARLY_WATCH_HOLD":

        return (
            "가격 Early Watch가 감지되었으나 "
            "현재 Inventory Position이 충분하여 "
            "추가 선행발주는 권고하지 않습니다."
        )


    if signal == "BASELINE_NO_FORECAST":

        return (
            f"가격 Forecast가 없어 "
            f"Baseline 정책 기준 "
            f"{recommended_qty:,.0f}kg 발주를 적용합니다."
        )


    return (
        f"추가 Price Risk 조정 없이 "
        f"Baseline 기준 {recommended_qty:,.0f}kg 발주를 권고합니다."
    )

In [10]:
def get_inventory_message(row):

    days_cover = row["ending_days_of_cover"]
    lead_time = row["lead_time_days"]
    inventory_position = row["inventory_position_kg"]

    return (
        f"이번 주 입고 반영 후 Inventory Position은 "
        f"{inventory_position:,.0f}kg이며, "
        f"발주·수요 반영 후 예상 Days of Cover는 "
        f"{days_cover:.1f}일입니다. "
        f"현재 적용 Lead Time은 {lead_time:.0f}일입니다."
    )

In [11]:
def create_scm_brief(row):

    risk_message = (
        get_risk_message(row)
    )

    inventory_message = (
        get_inventory_message(row)
    )

    action_message = (
        get_action_message(row)
    )


    brief = f"""
# SCM Weekly Decision Brief

### 기준 주차
{row["decision_week"].strftime("%Y-%m-%d")}

### 상품
{row["product_name"]} ({row["product_id"]})

---

## 1. Market Risk

- Current Price Index: {row["current_price_index"]:.3f}
- Forecast Price Index: {
    f'{row["forecast_price_index"]:.3f}'
    if pd.notna(row["forecast_price_index"])
    else "N/A"
}
- Current Risk: {row["current_risk_level"]}
- Forecast Risk: {
    row["forecast_risk_level"]
    if pd.notna(row["forecast_risk_level"])
    else "N/A"
}

{risk_message}

---

## 2. Inventory Status

- Available Inventory (after receipt): {row["beginning_inventory_kg"]:,.0f} kg
- Received This Week: {row["inbound_qty_kg"]:,.0f} kg
- Open Orders: {row["open_order_qty_kg"]:,.0f} kg
- Inventory Position: {row["inventory_position_kg"]:,.0f} kg
- Forecast Weekly Demand: {row["forecast_weekly_demand_kg"]:,.0f} kg
- Lead Time: {row["lead_time_days"]:.0f} days
- Safety Stock Policy: {row["safety_stock_days"]:.0f} days

{inventory_message}

---

## 3. Recommended Action

**{row["decision_signal"]}**

- Baseline Order: {row["baseline_order_qty_kg"]:,.0f} kg
- Recommended Order: {row["recommended_order_qty_kg"]:,.0f} kg
- Timing Adjustment: {row["order_timing_shift_kg"]:+,.0f} kg
- Price Risk Buffer: {row["risk_buffer_days"]:.0f} days

{action_message}

---

## 4. Decision Reason

{row["decision_reason"]}

---

### Data Scope

- Market Data: {row["market_data_scope"]}
- Enterprise SCM Data: {row["enterprise_data_scope"]}
- Decision Scope: {row["decision_scope"]}
- Policy Version: {row["policy_version"]}
- Data Quality: {row["data_quality_flag"]}
"""

    return brief.strip()

In [12]:
latest_brief = create_scm_brief(
    latest
)

print(latest_brief)

# SCM Weekly Decision Brief

### 기준 주차
2026-10-11

### 상품
양파 (P001)

---

## 1. Market Risk

- Current Price Index: 1.138
- Forecast Price Index: 1.128
- Current Risk: Watch
- Forecast Risk: Watch

현재 뚜렷한 가격 Risk 확대 신호는 없습니다.

---

## 2. Inventory Status

- Available Inventory (after receipt): 10,444 kg
- Received This Week: 3,500 kg
- Open Orders: 0 kg
- Inventory Position: 10,444 kg
- Forecast Weekly Demand: 4,948 kg
- Lead Time: 10 days
- Safety Stock Policy: 7 days

이번 주 입고 반영 후 Inventory Position은 10,444kg이며, 발주·수요 반영 후 예상 Days of Cover는 8.0일입니다. 현재 적용 Lead Time은 10일입니다.

---

## 3. Recommended Action

**BASELINE_REPLENISHMENT**

- Baseline Order: 7,000 kg
- Recommended Order: 7,000 kg
- Timing Adjustment: +0 kg
- Price Risk Buffer: 0 days

추가 Price Risk 조정 없이 Baseline 기준 7,000kg 발주를 권고합니다.

---

## 4. Decision Reason

기본 Safety Stock 및 Lead Time 기반 정기 보충발주

---

### Data Scope

- Market Data: PUBLIC_ACTUAL
- Enterprise SCM Data: SIMULATED
- Decision Scope: SIMULATION
- Policy Ver

In [13]:
advance_case = (
    decision[
        decision["decision_signal"]
        == "ADVANCE_PURCHASE"
    ]
    .sort_values("decision_week")
    .iloc[-1]
)

advance_case_brief = (
    create_scm_brief(
        advance_case
    )
)

print(
    advance_case_brief
)

# SCM Weekly Decision Brief

### 기준 주차
2026-09-27

### 상품
양파 (P001)

---

## 1. Market Risk

- Current Price Index: 1.040
- Forecast Price Index: 1.102
- Current Risk: Normal
- Forecast Risk: Watch

현재 가격은 정상 범위이나 다음 주 Watch 수준 상승 가능성이 감지되었습니다.

---

## 2. Inventory Status

- Available Inventory (after receipt): 9,979 kg
- Received This Week: 4,500 kg
- Open Orders: 0 kg
- Inventory Position: 9,979 kg
- Forecast Weekly Demand: 4,928 kg
- Lead Time: 7 days
- Safety Stock Policy: 7 days

이번 주 입고 반영 후 Inventory Position은 9,979kg이며, 발주·수요 반영 후 예상 Days of Cover는 6.8일입니다. 현재 적용 Lead Time은 7일입니다.

---

## 3. Recommended Action

**ADVANCE_PURCHASE**

- Baseline Order: 5,000 kg
- Recommended Order: 6,500 kg
- Timing Adjustment: +1,500 kg
- Price Risk Buffer: 2 days

Baseline 5,000kg 대비 1,500kg을 선행하여 총 6,500kg 발주를 권고합니다.

---

## 4. Decision Reason

Normal→Watch 가격 Risk 선행신호; 2일분 재고 Buffer 반영

---

### Data Scope

- Market Data: PUBLIC_ACTUAL
- Enterprise SCM Data: SIMULATED
- Decision Scope: SI

In [14]:
BRIEF_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "briefs"
)

BRIEF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [15]:
latest_path = (
    BRIEF_DIR
    / (
        "scm_brief_"
        + latest[
            "decision_week"
        ].strftime("%Y%m%d")
        + ".md"
    )
)

latest_path.write_text(
    latest_brief,
    encoding="utf-8",
)

print(latest_path)

/Users/cindy/meanvelop/retailops-ax/outputs/briefs/scm_brief_20261011.md


In [16]:
advance_case_path = (
    BRIEF_DIR
    / (
        "scm_brief_advance_case_"
        + advance_case[
            "decision_week"
        ].strftime("%Y%m%d")
        + ".md"
    )
)

advance_case_path.write_text(
    advance_case_brief,
    encoding="utf-8",
)

print(advance_case_path)

/Users/cindy/meanvelop/retailops-ax/outputs/briefs/scm_brief_advance_case_20260927.md
